## Problem 1: Spectral Decomposition of Orthogonal Matrices (6 = 1 + 2 + 3 points)

The proof of the Spectral Decomposition of an orthogonal matrix (see the reader or appendix below) uses induction, which shows great similarities with the recursive problems discussed earlier. Your task will be to implement the Spectral decomposition of orthogonal matrices in Python, the procedure of which should be in line with the proof of the spectral theorem. 

Implement the functions `orthonormal_basis`, `base_case` and `spectral_decomposition` below. 

Important instructions:

1. For `orthonormal_basis` it might be useful to think about how you can use the built-in QR factorization `np.linalg.qr` using the 'complete' mode, you can find its documentation [here](https://numpy.org/doc/stable/reference/generated/numpy.linalg.qr.html). Note that this approach might alter the sign of the original columns, do not worry about this!

2. Each iteration use the first eigenvalue and eigenvector from the eVals and eVecs to get results that are consistent with the provided test cases. Mathematically it does not matter which one you choose, but it might be easier to check your work if you stick to this rule. 

3. You can check if an eigenvalue is real or complex using the methods `np.isreal` or `np.iscomplex`, see the documentation [here](https://numpy.org/doc/stable/reference/generated/numpy.isreal.html). **However**, the eigenvalue/eigenvector function `np.linalg.eig` is only an approximation of the actual eigenvalues/eigenvectors and might make rounding errors that makes real eigenvalues be treated as complex eigenvalues. For example the matrix $\begin{bmatrix} 3 & 0 & 0 \\ 1 & 1 & 2 \\ 1 & -2 & 5 \end{bmatrix}$ only has eigenvalue $\lambda=3$, but Python will return $\lambda=3+0.00000003i, \lambda=3-0.00000003i$ and $\lambda=3$. So **instead** of using `np.isreal` to check if an eigenvalue is real, you should use `np.isclose` to check if the imaginary part of an eigenvalue is close to zero, which you can find using `np.imag`. 

5. When your matrix has complex eigenvalues, the eVecs matrix containing eigenvectors will show real numbers as complex numbers. That is, the number 3 will be treated as 3 + 0j. To make everything print nicely (which makes it easier to verify the test cases), convert the eigenvectors to display real values using `np.real`.

6. You should skip the final step in the proof, namely that of reshuffling the columns of $Q$ to make the matrix $B$ be in the form ($\star$). 

In [13]:
def orthonormal_basis(A):
    """
    Computes the matrix Q whose first k columns form an orthonormal basis for
    the image of A, and extends this to an orthonormal basis of R^n so that
    Q is an orthogonal matrix
    
    Parameters
    ----------
        A (ndarray): n x k matrix with linearly independent columns
        
    Returns
    -------
        Q (ndarray): n x n orthogonal matrix.
    
    """
    Q, R = la.qr(A, mode='complete')
    return Q
    

In [14]:
def base_case(A):
    """
    Computes the matrices Q and B in the Spectral decomposition of A, 
    that is A = Q B Q^T, in the special casess that A is either
    a 1x1 or 2x2 matrix. Implementation should be in line with that 
    as discussed in the proof of the reader
    
    Parameters
    ----------
        A (ndarray): an arbitrary 1x1 or 2x2 orthogonal matrix with 
                     real entries.
        
    Returns
    -------
        Q (ndarray): orthogonal matrix.
        B (ndarray): matrix in canonical form (⋆) such that QBQ^T = A.
    
    """
    n = A.shape[0]
    eVals, eVecs = la.eig(A)  # The eig function returns normalized eigenvectors
    
    idx = 0
    if abs(eVals[0] + 1) < 1e-6 and np.any(abs(eVals - 1) < 1e-6):
        idx = np.argmax(abs(eVals - 1) < 1e-6)

    if n == 1:
        Q = np.eye(1)
        B = A
        return (Q, B)

    if abs(np.imag(eVals[idx])) < 1e-6:
        u1 = np.real(eVecs[:, idx:idx+1])
        Q = orthonormal_basis(u1)
    else:
        w = eVecs[:, idx]
        u1 = math.sqrt(2) * np.imag(w) / la.norm(w)
        u2 = math.sqrt(2) * np.real(w) / la.norm(w)
        Q = np.column_stack([u1, u2])

    B = Q.T @ A @ Q
    return (Q, B)


In [22]:
def spectral_decomposition(A):
    """
    Computes the matrices Q and B in a Spectral decomposition of A, 
    that is A = Q B Q^T. Implementation should be in line with that
    as discussed in the proof of the textbook.
    
    Parameters
    ----------
        A (ndarray): an orthogonal n x n matrix.
        
    Returns
    -------
        Q (ndarray): orthogonal n x n matrix.
        B (ndarray): matrix in canonical form (⋆) such that QBQ^T = A.
    
    """
    n = A.shape[0]
    
    if n <= 2:
        Q, B = base_case(A)
        return(Q, B)
    
    eVals, eVecs = la.eig(A)

    idx = 0
    if abs(eVals[0] + 1) < 1e-6 and np.any(abs(eVals - 1) < 1e-6):
        idx = np.argmax(abs(eVals - 1) < 1e-6)

    if abs(np.imag(eVals[idx])) < 1e-6:
        k = 1
        u1 = np.real(eVecs[:, idx:idx+1])
        Q1 = orthonormal_basis(u1)
    else:
        k = 2
        w = eVecs[:, idx]
        u1 = math.sqrt(2) * np.imag(w) / la.norm(w)
        u2 = math.sqrt(2) * np.real(w) / la.norm(w)
        Q1 = orthonormal_basis(np.column_stack([u1, u2]))

    A1 = (Q1.T @ A @ Q1)[k:, k:]
    P, _ = spectral_decomposition(A1)

    Q2 = np.eye(n)
    Q2[k:, k:] = P
    Q = Q1 @ Q2
    B = Q.T @ A @ Q

    return (Q, B)
    

In [23]:
# You can use this code cell to play around with your functions to make sure
# they do what they are intended to do, i.e. to debug your code. 


In [24]:
# Test case for orthonormal_basis
# Note that the sign of the first column has changed, don't worry about this!
np.set_printoptions(precision=2, suppress=True)  # Print zeros and decimals nicely

A = np.array([[0, 1], [1, 1], [2, 1], [2, 1]])
Q = orthonormal_basis(A)
print("A = \n", A, "\n")
print("Q = \n", Q)

A = 
 [[0 1]
 [1 1]
 [2 1]
 [2 1]] 

Q = 
 [[ 0.   -0.9  -0.3  -0.3 ]
 [-0.33 -0.4   0.6   0.6 ]
 [-0.67  0.1   0.35 -0.65]
 [-0.67  0.1  -0.65  0.35]]


Expected output:

    A = 
     [[0 1]
     [1 1]
     [2 1]
     [2 1]] 

    Q = 
     [[ 0.   -0.9  -0.3  -0.3 ]
     [-0.33 -0.4   0.6   0.6 ]
     [-0.67  0.1   0.35 -0.65]
     [-0.67  0.1  -0.65  0.35]]

In [25]:
# Test cases for base_case
np.set_printoptions(precision=2, suppress=True)  # Print zeros and decimals nicely

A1 = np.array([[-1]])
Q1, B1 = base_case(A1)
print("Q1 =\n", Q1, "\nB1 =\n", B1, "\n")

A2 = np.array([[0, 1], [1, 0]])
Q2, B2 = base_case(A2)
print("Q2 =\n", Q2, "\nB2 =\n", B2, "\n")

A3 = 1/math.sqrt(2) * np.array([[1, 1], [-1, 1]])
Q3, B3 = base_case(A3)
print("Q3 =\n", Q3, "\nB2 =\n", B3, "\n")

Q1 =
 [[1.]] 
B1 =
 [[-1]] 

Q2 =
 [[-0.71 -0.71]
 [-0.71  0.71]] 
B2 =
 [[ 1. -0.]
 [-0. -1.]] 

Q3 =
 [[0. 1.]
 [1. 0.]] 
B2 =
 [[ 0.71 -0.71]
 [ 0.71  0.71]] 



Expected output:

    Q1 =
     [[1.]] 
    B1 =
     [[-1]] 

    Q2 =
     [[-0.71 -0.71]
     [-0.71  0.71]] 
    B2 =
     [[ 1. -0.]
     [-0. -1.]] 

    Q3 =
     [[0. 1.]
     [1. 0.]] 
    B2 =
     [[ 0.71 -0.71]
     [ 0.71  0.71]] 

In [26]:
# Test cases for spectral_decomposition
np.set_printoptions(precision=2, suppress=True)  # Print zeros and decimals nicely

A1 = 1/3*np.array([[2,1,-2],[1,2,2],[2,-2,1]])
Q1, B1 = spectral_decomposition(A1)
print("Q1 =\n", Q1, "\nB1 =\n",B1, "\n")

A2 = np.array([[0, 0, 1], [1, 0, 0], [0, 1, 0]])
Q2, B2 = spectral_decomposition(A2)
print("Q2 =\n", Q2, "\nB2 =\n",B2, "\n")

A3 = 1/3*np.array([[2,1,-2],[1,2,2],[-2,2,-1]])
Q3, B3 = spectral_decomposition(A3)
print("Q3 =\n", Q3, "\nB3 =\n",B3, "\n")

A4 = 1/2*np.array([[1,1,1,1],[1,1,-1,-1],[1,-1,1,-1],[1,-1,-1,1]])
Q4, B4 = spectral_decomposition(A4)
print("Q4 =\n", Q4, "\nB4 =\n",B4, "\n")

A5 = np.array([[1,0,0,0],[0,0,1,0],[0,-1,0,0],[0,0,0,1]])
Q5, B5 = spectral_decomposition(A5)
print("Q5 =\n", Q5, "\nB5 =\n",B5, "\n")

Q1 =
 [[-0.71 -0.   -0.71]
 [-0.71 -0.    0.71]
 [-0.    1.   -0.  ]] 
B1 =
 [[ 1.   -0.    0.  ]
 [ 0.    0.33 -0.94]
 [ 0.    0.94  0.33]] 

Q2 =
 [[ 0.   -0.82  0.58]
 [ 0.71  0.41  0.58]
 [-0.71  0.41  0.58]] 
B2 =
 [[-0.5  -0.87  0.  ]
 [ 0.87 -0.5   0.  ]
 [-0.    0.    1.  ]] 

Q3 =
 [[-0.91  0.    0.41]
 [-0.18 -0.89 -0.41]
 [ 0.37 -0.45  0.82]] 
B3 =
 [[ 1. -0. -0.]
 [ 0.  1.  0.]
 [-0.  0. -1.]] 

Q4 =
 [[-0.87  0.    0.   -0.5 ]
 [-0.29 -0.82 -0.    0.5 ]
 [-0.29  0.41 -0.71  0.5 ]
 [-0.29  0.41  0.71  0.5 ]] 
B4 =
 [[ 1.  0.  0. -0.]
 [-0.  1.  0.  0.]
 [ 0. -0.  1.  0.]
 [-0.  0.  0. -1.]] 

Q5 =
 [[0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 0. 1.]] 
B5 =
 [[ 0. -1.  0.  0.]
 [ 1.  0.  0.  0.]
 [ 0.  0.  1.  0.]
 [ 0.  0.  0.  1.]] 



Expected output:

    Q1 =
     [[-0.71  0.71  0.  ]
     [-0.71 -0.71  0.  ]
     [ 0.   -0.    1.  ]] 
    B1 =
     [[ 1.    0.   -0.  ]
     [-0.    0.33 -0.94]
     [ 0.    0.94  0.33]] 

    Q2 =
     [[-0.71 -0.41  0.58]
     [ 0.71 -0.41  0.58]
     [ 0.    0.82  0.58]] 
    B2 =
     [[-0.5  -0.87  0.  ]
     [ 0.87 -0.5  -0.  ]
     [ 0.    0.    1.  ]] 

    Q3 =
     [[-0.91  0.    0.41]
     [-0.18 -0.89 -0.41]
     [ 0.37 -0.45  0.82]] 
    B3 =
     [[ 1. -0. -0.]
     [ 0.  1.  0.]
     [-0.  0. -1.]] 

    Q4 =
     [[-0.87  0.   -0.   -0.5 ]
     [-0.29 -0.82 -0.    0.5 ]
     [-0.29  0.41 -0.71  0.5 ]
     [-0.29  0.41  0.71  0.5 ]] 
    B4 =
     [[ 1. -0.  0. -0.]
     [-0.  1.  0.  0.]
     [-0.  0.  1. -0.]
     [-0.  0. -0. -1.]] 

    Q5 =
     [[0. 0. 1. 0.]
     [0. 1. 0. 0.]
     [1. 0. 0. 0.]
     [0. 0. 0. 1.]] 
    B5 =
     [[ 0. -1.  0.  0.]
     [ 1.  0.  0.  0.]
     [ 0.  0.  1.  0.]
     [ 0.  0.  0.  1.]]

In [27]:
# AUTOGRADER FOR orthonormal_basis, BEWARE OF POTENTIAL HIDDEN TEST CASES
# Note that this not an exhaustive check for correctness.
A = np.array([[1], [0]])
Q = orthonormal_basis(A)
assert np.allclose(Q.T @ Q, np.eye(2))
assert np.allclose(Q[:,1:2].T @ A, np.zeros((1,1)))

A1 = np.array([[1], [2], [3]])
Q1 = orthonormal_basis(A1)
assert np.allclose(Q1.T @ Q1, np.eye(3))
assert np.allclose(Q1[:,1:3].T @ A1, np.zeros((2,1)))

A2 = np.array([[0, 1], [1, 1], [2, 1], [2, 1]])
Q2 = orthonormal_basis(A2)
assert np.allclose(Q2.T @ Q2, np.eye(4))
assert np.allclose(Q2[:,2:4].T @ A2, np.zeros((2,2)))


In [28]:
# AUTOGRADER FOR base_case, BEWARE OF POTENTIAL HIDDEN TEST CASES
# Note that this not an exhaustive check for correctness.
A1 = np.array([[-1]])
Q1, B1 = base_case(A1)
assert np.allclose(Q1, np.eye(1))
assert np.allclose(B1, A1)

A2 = np.array([[0, 1], [1, 0]])
Q2, B2 = base_case(A2)
assert np.allclose(Q2, 1/math.sqrt(2) * np.array([[-1, -1], [-1, 1]]))
assert np.allclose(B2, np.array([[1,0],[0,-1]]))

A3 = 1/math.sqrt(2) * np.array([[1, 1], [-1, 1]])
Q3, B3 = base_case(A3)
assert np.allclose(Q3, np.array([[0,1],[1,0]]))
assert np.allclose(B3, 1/math.sqrt(2) * np.array([[1,-1],[1,1]]))


In [30]:
# AUTOGRADER FOR spectral_decomposition, BEWARE OF POTENTIAL HIDDEN TEST CASES
# Note that this not an exhaustive check for correctness.
A1 = 1/3*np.array([[2,1,-2],[1,2,2],[2,-2,1]])
Q1, B1 = spectral_decomposition(A1)
assert np.allclose(Q1.T @ Q1, np.eye(3))
assert np.allclose(B1, np.array([[1, 0, 0], [0, 1/3, -math.sqrt(8/9)], [0, math.sqrt(8/9), 1/3]]))
assert np.allclose(Q1 @ B1 @ Q1.T, A1)

A2 = np.array([[0, 0, 1], [1, 0, 0], [0, 1, 0]])
Q2, B2 = spectral_decomposition(A2)
assert np.allclose(Q2.T @ Q2, np.eye(3))
assert np.allclose(B2, np.array([[-0.5, -math.sqrt(3/4), 0], [math.sqrt(3/4), -0.5, 0], [0, 0, 1]]))
assert np.allclose(Q2 @ B2 @ Q2.T, A2)

A3 = 1/3*np.array([[2,1,-2],[1,2,2],[-2,2,-1]])
Q3, B3 = spectral_decomposition(A3)
assert np.allclose(Q3.T @ Q3, np.eye(3))
assert np.allclose(B3, np.array([[1, 0, 0], [0, 1, 0], [0, 0, -1]]))
assert np.allclose(Q3 @ B3 @ Q3.T, A3)

A4 = 1/2*np.array([[1,1,1,1],[1,1,-1,-1],[1,-1,1,-1],[1,-1,-1,1]])
Q4, B4 = spectral_decomposition(A4)
assert np.allclose(Q4.T @ Q4, np.eye(4))
assert np.allclose(B4, np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, -1]]))
assert np.allclose(Q4 @ B4 @ Q4.T, A4)


## Appendix: Proof of Spectral Decomposition

We prove by induction on $n$ that a real symmetric $n \times n$ matrix is orthogonally similar to a diagonal matrix, i.e. there exists an orthogonal matrix $Q$ such that $Q^T A Q = D$ is diagonal. 

**Base case:** For a $1 \times 1$ matrix $A$ we can set $Q = [1]$ and the statement is trivially true for $n=1$. 

**Induction step:** Let $A$ be a $(n+1) \times (n+1)$ real symmetric matrix where $n \geq 1$ and assume that the claim is true for all $1 \leq k \leq n$. Let $\lambda_1$ be an eigenvalue of $A$, which exists by the fundamental theorem of algebra. Let $u_1$ be a unit eigenvector associated with $\lambda_1$. We can extend $\{u_1\}$ to an orthonormal basis $\{u_1, u_2, \dots, u_{n+1}\}$ of $\mathbb{R}^{n+1}$ and form the orthogonal matrix
    $$Q_1 = \begin{bmatrix}
        u_1 & u_2 & \dots & u_{n+1} 
    \end{bmatrix}.$$
the first column of $Q^T A Q$ is
$$Q^T A Q e_1 = Q^T A u_1 = Q^T(\lambda_1 u_1) = \lambda_1 Q^{-1} u_1 = \lambda_1 e_1.$$
Secondly we also note that $Q^T A Q$ is symmetric, since
$$(Q^T A Q)^T = Q^T A^T (Q^T)^T = Q^T A Q,$$
where we used that $A$ is symmetric. So $Q_1^T A Q_1$ will be a matrix of block form
    $$Q_1^T A Q_1 = \begin{bmatrix}
        \lambda_1 & 0 \\
        0 & A_1
    \end{bmatrix},$$
where $A_1$ is a real symmetric $n \times n$ matrix. By the induction hypothesis applied to $A_1$ there exists an orthogonal matrix $R$ such that $R^T A_1 R = D$ is a diagonal matrix. If we set
    $$Q_2 = \begin{bmatrix}
        1 & 0 \\
        0 & R
    \end{bmatrix} \quad \text{and} \quad Q = Q_1 Q_2,$$
then both $Q_2$ and $Q$ are $(n+1) \times (n+1)$ orthogonal matrices. Moreover
    $$\begin{align*}
        Q^T A Q = Q_2^T Q_1^T A Q_1 Q_2= 
        \begin{bmatrix}
            1 & 0 \\
            0 & R^T
        \end{bmatrix} \begin{bmatrix}
            \lambda_1 & 0 \\
            0 & A_1
        \end{bmatrix}\begin{bmatrix}
            1 & 0 \\
            0 & R
        \end{bmatrix} = \begin{bmatrix}
            \lambda_1 & 0 \\
            0 & R^T A_1 R
        \end{bmatrix} = \begin{bmatrix}
        \lambda_1 & 0 \\
        0 & D
    \end{bmatrix},
    \end{align*}$$
Hence it follows that there exists an orthogonal matrix $Q$ that brings $A$ into diagonal form, proving our claim. 

**Theorem (Spectral theorem for orthogonal matrices)** Let $A$ be an orthogonal $n \times n$ matrix. Then there exists an orthogonal matrix $Q$ such that 
\begin{equation}
   Q^T A Q = \begin{bmatrix}
   \pm 1 & & & & & \\
   & \ddots & & & & \\
   & & \pm 1 & & & \\
   & & & R_1 & & \\
   & & & & \ddots & \\
   & & & & & R_k
\end{bmatrix} \tag{$\star$}
\end{equation}
where $R_1, \dots, R_m$ are rotation matrices of the form
$$\begin{bmatrix}
   \cos(\theta) & -\sin(\theta) \\
   \sin(\theta) & \cos(\theta)
\end{bmatrix},$$
where $\cos(\theta) = \text{Re}(\lambda)$ for any complex eigenvalue $\lambda$ of $A$. 

**Proof:** Similar to the proof of the spectral theorem for symmetric matrices, we prove the claim again by induction on the dimension $n$. This time around we need to distinguish several base cases, where we first show the claim is true for all $1 \times 1$ and $2 \times 2$ matrices. 

**Base case (1):** An orthogonal $1 \times 1$ matrix $A$ must be of the form $\begin{bmatrix}
        \pm 1
    \end{bmatrix}$. So taking $Q = \begin{bmatrix}
        1
    \end{bmatrix}$ yields that $Q^T A Q = \begin{bmatrix}
        \pm 1
    \end{bmatrix}$ and the statement of the theorem is trivially true for $n=1$.

**Base case (2.1):** For an orthogonal $2 \times 2$ matrix $A$, let's first assume that the eigenvalues $\lambda_1$ and $\lambda_2$ of $A$ are both real, in which case we know that $\lambda_1, \lambda_2 \in \{1, -1\}$. Let $u_1$ be a unit eigenvector of $A$ corresponding to $\lambda_1$, and let $u_2$ be any unit vector that is orthogonal to $u_1$. If we set $Q = \begin{bmatrix}
    u_1 & u_2
\end{bmatrix}$, then $Q$ is an orthogonal matrix. The first column of $Q^T A Q$ is
$$Q^T A Q e_1 = Q^T A u_1 = Q^T (\lambda_1 u_1) = \lambda_1 Q^{-1} u_1 = \lambda_1 e_1.$$
Moreover, note that $Q^T A Q$ is a product of orthogonal matrices, so that $Q^T A Q$ is itself also orthogonal. Since the columns of an orthogonal matrix must be orthonormal, we deduce that the second column of $Q^T A Q$ must be a unit vector orthogonal to $\lambda_1 e_1 = \pm e_1$, i.e. the $Q^T A Q = \pm e_2$. We conclude that
$$Q^T A Q = \begin{bmatrix}
    \pm 1 & O \\ 
    O & \pm 1
\end{bmatrix}.$$
**Base case (2.2):** For an orthogonal $2 \times 2$ matrix $A$, let's now assume that the eigenvalues $\lambda_1$ and $\lambda_2$ are non-real, and let $w = x + iy$ be an eigenvector corresponding to $\lambda_1$. Setting $Q = \begin{bmatrix}
    \frac{\sqrt{2} y}{\|w\|} & \frac{\sqrt{2} x}{\|w\|}
\end{bmatrix}$, we know by Theorem ... that $Q$ is orthogonal and 
$$Q^T A Q = \begin{bmatrix}
    \cos(\theta) & -\sin(\theta) \\
    \sin(\theta) & \cos(\theta)
\end{bmatrix},$$
where $\cos(\theta) = \text{Re}(\lambda_1)$. This shows that the statement of the theorem is also true for $n=2$. 

**Induction hypothesis:** Suppose the claim is true for all $1 \leq k \leq n$, that is for all orthogonal $k \times k$ matrices $A$ there exists an orthogonal matrix $Q$ such that $Q^T A Q$ is of the form $(\star)$.

**Induction step:** Let $A$ be an orthogonal $(n+1) \times (n+1)$ matrix and let $\lambda_1$ be an eigenvalue of $A$, which exists by the fundamental theorem of algebra. We distinguish two cases again.

First, assume $\lambda_1$ is real (so $\lambda_1 \in \{1, -1\}$) and let $u_1$ be an associated unit eigenvector. We can extend $\{u_1\}$ to an orthonormal basis $\{u_1, u_2, \dots, u_{n+1}\}$ of $\mathbb{R}^{n+1}$ and form the orthogonal matrix
$$Q_1 = \begin{bmatrix}
        u_1 & u_2 & \dots u_{n+1}
    \end{bmatrix}.$$
As in the $2 \times 2$ case we see that the first column of $Q_1^T A Q_1$ will be $\lambda_1 e_1$ and moreover $Q_1^T A Q_1$ is also an orthogonal matrix. This means that $Q_1^T A Q_1$ is of the form
$$\begin{bmatrix}
    \pm 1 & O \\
    O & A_1
\end{bmatrix},$$
where $A_1$ is some $n \times n$ orthogonal matrix. 

Now assume that $\lambda_1$ is non-real and let $w = x + iy$ be an associated eigenvector of $A$. Let $u_1 = \frac{\sqrt{2} y}{\|w\|}$ and $u_2 = \frac{\sqrt{2} x}{\|w\|}$, which we know are orthonormal by \autoref{thm:rotation}. Extend $\{u_1, u_2\}$ to an orthonormal basis $\{u_1, u_2, \dots, u_{n+1}\}$ of $\mathbb{R}^{n+1}$ and define the orthogonal matrix
$$Q_1 = \begin{bmatrix}
    u_1 & u_2 & \dots & u_{n+1}
\end{bmatrix}.$$
By Theorem ... we know that $Q^T A Q$ is of the form
$$Q_1^T A Q_1 = \begin{bmatrix}
        \cos(\theta) & -\sin(\theta) & X \\
        \sin(\theta) & \cos(\theta) & X \\
        O & O & A_1
    \end{bmatrix},$$
where $X$ is a $2 \times (n-2)$ matrix and $A_1$ is an $(n-1) \times (n-1)$ matrix. Since $Q_1^T A Q_1$ is orthogonal we deduce that $X = 0$ and $A_1$ is an orthogonal matrix, so that
$$Q_1^T A Q_1 = \begin{bmatrix}
        \cos(\theta) & -\sin(\theta) & O \\
        \sin(\theta) & \cos(\theta) & O\\
        O & O & A_1
    \end{bmatrix}.$$

So, in general, we have shown that there exists an orthogonal $(n+1) \times (n+1)$ matrix $Q_1$ such that $Q_1^T A Q_1$ is of the form
$$Q_1^T A Q_1 = \begin{bmatrix}
    B & O \\
    O & A_1
\end{bmatrix},$$
where
$$\begin{align}
    B &= \begin{bmatrix}
        \pm 1
    \end{bmatrix} \text{ and $A_1$ is an orthogonal $n \times n$ matrix} && \text{if $\lambda_1$ is real}; \\
    B &= \begin{bmatrix}
        \cos(\theta) & -\sin(\theta) \\
        \sin(\theta) & \cos(\theta)
    \end{bmatrix} \text{ and $A_2$ is an orthogonal $(n-1) \times (n-1)$ matrix} && \text{if $\lambda_1$ is non-real}.
\end{align}$$
In either case, by the induction hypothesis applied to $A_1$, there exists an orthogonal matrix $P$ such that $P^T A_1 P$ is of the form $(\star)$, i.e.
$$P^T A_1 P = \begin{bmatrix}
   \pm 1 & & & & & \\
   & \ddots & & & & \\
   & & \pm 1 & & & \\
   & & & R_1 & & \\
   & & & & \ddots & \\
   & & & & & R_k
\end{bmatrix}.$$
If we set
$$Q_2 = \begin{bmatrix}
   I & O \\
   O & P
\end{bmatrix} \quad \text{and} \quad Q = Q_1 Q_2,$$
then both $Q_2$ and $Q$ are $(n+1) \times (n+1)$ orthogonal matrices. Morover,
$$Q^T A Q = Q_2^T \left( Q_1^T A Q_1\right) Q_2 = \begin{bmatrix}
   I & O \\
   O & P^T
\end{bmatrix} \begin{bmatrix}
   B & O \\
   O & A_1
\end{bmatrix} \begin{bmatrix}
   I & O \\
   O & P
\end{bmatrix} = \begin{bmatrix}
   B & O \\
   O & P^T A_1 P
\end{bmatrix}.$$
Recalling that $B$ and $P^T A_1 P$ are of the form $(\star)$, after reordering the columns of $Q$, we find that $Q^T A Q$ is of the desired form $(\star)$. This completes the induction step, and we conclude that, for all $n \geq 1$, an orthogonal $n \times n$ matrix is orthogonally similar to $(\star)$.